## Overview 
The Clinical Effectiveness Team at MediTrack has noticed varying success rates and costs across different treatment approaches. They've approached you, their trusted data scientist, with a critical question: "Can we quantitatively determine which treatment approaches are most cost-effective while maintaining quality care?"

## Your Challenge 
MediTrack needs to:
- Compare the effectiveness and costs of different treatment approaches
- Determine if certain treatments are more cost-effective for specific conditions
- Provide statistical evidence to support treatment recommendations
- Create a repeatable analysis framework for future treatment comparisons

This analysis will help MediTrack:
- Optimize treatment recommendations
- Control healthcare costs while maintaining quality
- Support evidence-based decision making
- Guide insurance coverage policies

## Learning Outcomes 
- Design and implement A/B tests for healthcare treatment comparison
- Apply statistical hypothesis testing to evaluate treatment effectiveness
- Create compelling visualizations of treatment outcomes
- Develop data-driven recommendations for clinical decision-making

## Dataset Information
Building on your previous work with MediTrack's integrated database, you'll focus on the medical_visits dataset, which now includes:
- Detailed treatment information
- Associated doctor fees
- Patient outcomes
- Visit date

This represents real-world data from MediTrack's network of healthcare providers, though anonymized for privacy.

## Activities
### Activity 1: Initial Data Investigation 
As MediTrack's data scientist, your first task is to understand the current treatment landscape.

<b>Step 1:</b> Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene, ttest_ind
from pathlib import Path

<b>Step 2:</b> Adjust paths to import data

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[1]
DATA_DIR = BASE_DIR / "data"
medical_visits_path = DATA_DIR / "medical_visits_AB.xlsx"

<b>Step 3:</b> Import data

In [ ]:
df = pd.read_excel('medical_visits_AB.xlsx')

# Examine data structure
print("Dataset Info:")
print(df.info())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Basic statistics
print("\nBasic Statistics:")
print(df.describe())

# Treatment distribution
print("\nTreatment Distribution:")
print(df['treatment'].value_counts())

### Activity 2: Treatment Comparison Framework 
The Clinical Effectiveness Team needs a robust method to compare treatments.

<b>Step 1:</b> Define treatment groups for comparison

In [ ]:
# Extract treatment groups
therapy_group = df[df['treatment'] == 'Therapy']['doctor_fee']
surgery_group = df[df['treatment'] == 'Surgery']['doctor_fee']

<b>Step 2:</b> Verify statistical assumptions

In [ ]:
def check_assumptions(group1, group2):
    """Check statistical assumptions for treatment comparison."""
    
    # Test for normality
    _, p_norm1 = shapiro(group1)
    _, p_norm2 = shapiro(group2)
    normality_met = (p_norm1 > 0.05) and (p_norm2 > 0.05)
    
    # Test for equal variance
    _, p_var = levene(group1, group2)
    equal_variance = p_var > 0.05
    
    results = {
        'normality_met': normality_met,
        'equal_variance': equal_variance,
        'parametric_appropriate': normality_met and equal_variance
    }
    
    return results

In [ ]:
# Check assumptions
assumptions = check_assumptions(therapy_group, surgery_group)
print("Statistical Assumptions Check:")
print(assumptions)

### Activity 3: Conducting the A/B Test

In [ ]:
def perform_statistical_test(group1, group2, assumptions_met=True):
    """Conduct appropriate statistical test based on assumptions."""
    
    if assumptions_met:
        # Use Student's t-test (assumes equal variances)
        stat, p_value = ttest_ind(group1, group2)
        test_type = "Student's t-test (equal_var=True)"
    else:
        # Use Welch's t-test (does not assume equal variances)
        stat, p_value = ttest_ind(group1, group2, equal_var=False)
        test_type = "Welch’s t-test (equal_var=False)"
    
    return {
        'test_type': test_type,
        'statistic': stat,
        'p_value': p_value
    }

# Perform statistical test
test_results = perform_statistical_test(
    therapy_group,
    surgery_group,
    assumptions['parametric_appropriate']
)

print("\nStatistical Test Results:")
print(test_results)

In [ ]:
# Visualization of Results 

def visualize_results(df):
    """Create visualization for treatment comparison."""
    
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='treatment', y='doctor_fee', data=df)
    plt.title('Treatment Costs Comparison')
    plt.ylabel('Doctor Fee ($)')
    plt.xlabel('Treatment')
    plt.tight_layout()
    plt.show()

# Create visualization
visualize_results(df)